In [ ]:
# !pip install datasets

In [ ]:
# 패키지 준비

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from datasets import load_dataset


In [ ]:
# 데이터 준비

# NSMC (네이버 영화 리뷰 감성 분석) 데이터 활용 예시
# 실제 사용 시: https://github.com/e9t/nsmc 에서 다운로드
# 또는 datasets 라이브러리: pip install datasets

# nsmc = load_dataset("nsmc")
# DatasetDict({
#     train: Dataset({features: ['id', 'document', 'label'], num_rows: 150000}),
#     test:  Dataset({features: ['id', 'document', 'label'], num_rows: 50000})
# })

nsmc = load_dataset(
    "csv",
    data_files={
        "train": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
        "test": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"
    },
    sep="\t"
)

# 데이터 확인
# print(nsmc)
# print(nsmc['train'].column_names)
# print(nsmc['train'][0])
# print(nsmc['train'][0]['document'], nsmc['train'][0]['label'])

class NSMCDataset(Dataset):
    """네이버 영화 리뷰 감성 분류 데이터셋"""
    
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data[idx]
        text = str(row['document']) if row['document'] else ""
        label = int(row['label'])
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length, # 모든 문장은 128개 토큰으로 구성
            padding='max_length',
            truncation=True,
            return_tensors='pt' # pytorch 형식으로 반환
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 데이터로더 생성
MODEL_NAME = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = NSMCDataset(nsmc['train'], tokenizer)
test_dataset  = NSMCDataset(nsmc['test'], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=4)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=4)

print(f"훈련: {len(train_dataset):,}개 / 테스트: {len(test_dataset):,}개")

print(train_dataset[0])

훈련: 150,000개 / 테스트: 50,000개
{'input_ids': tensor([   2, 1376,  831, 2604,   18,   18, 4229, 9801, 2075, 2203, 2182, 4243,
           3,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
  

In [ ]:
# 모델 설계

# Trainer를 사용하면 학습 루프를 직접 작성하지 않아도 됨
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import load_metric

# AutoModelForSequenceClassification: 분류 헤드가 내장된 모델
model_hf = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    evaluation_strategy="epoch",  # 에폭마다 평가
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    fp16=torch.cuda.is_available(),  # GPU 있으면 혼합 정밀도
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

trainer = Trainer(
    model=model_hf,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# 단 한 줄로 훈련 시작
trainer.train()
trainer.evaluate()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


모델 파라미터: 110,618,882
